# LLM Evaluation Pipeline – Separat vs. Kombiniert
Vergleicht zwei Prompt-Strategien über mehrere Modelle:
- **Separat**: drei einzelne Prompts pro Aufgabe
- **Kombiniert**: ein Prompt für alle drei Aufgaben

Metriken: Accuracy, Precision, Recall, F1-Score

## 1. Installation

In [1]:
# Erste Zelle im Colab-Notebook — GPU prüfen
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
from google.colab import userdata
import os
import pathlib as path
from huggingface_hub import login

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
login(token=userdata.get("HF_token"))
"""
key_path = Path("/home/theo/PycharmProjects/Masterprojekt-Chatbots/API_KEYS/openai_key.txt")
if key_path.exists():
    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()
    print("OpenAI API Key gesetzt")
else:
    print("Key-Datei nicht gefunden")7
"""

'\nkey_path = Path("/home/theo/PycharmProjects/Masterprojekt-Chatbots/API_KEYS/openai_key.txt")\nif key_path.exists():\n    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()\n    print("OpenAI API Key gesetzt")\nelse:\n    print("Key-Datei nicht gefunden")7\n'

In [ ]:
!pip install -q langchain langchain-openai langchain-huggingface langchain-community
!pip install -q transformers torch accelerate scikit-learn pandas pydantic tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()  # öffnet Datei-Dialog

import pandas as pd
df = pd.read_csv("gs_merged.csv")

Saving gs_merged.csv to gs_merged (1).csv


## 2. Datensatz laden

In [ ]:
import pandas as pd
from pathlib import Path

#repo = Path(".").resolve().parent
#df   = pd.read_csv(repo / "data/processed/control/gs_merged.csv")

TASK_MAP = {
    1: "Informationssuche und Verständnis",
    2: "Schreiben und Textarbeit",
    3: "Praktische Unterstützung und Strukturierung",
    4: "Technische und analytische Unterstützung",
    5: "Lernen und Prüfungsvorbereitung",
}
SENTIMENT_MAP = {1: "Unfreundlich", 2: "Neutral", 3: "Freundlich"}
CRITICAL_MAP  = {0: "Nein", 1: "Ja"}

df_chats = (
    df.groupby("chat_id")
    .agg(
        text            = ("content", lambda x: "\n".join(
                            f"User_Nachricht_{i+1}: {msg}"
                            for i, msg in enumerate(x))),
        label_task      = ("task",      "first"),
        label_sentiment = ("sentiment", "first"),
        label_critical  = ("critical",  "first"),
    )
    .reset_index()
)

df_chats["label_task"]      = df_chats["label_task"].map(TASK_MAP)
df_chats["label_sentiment"] = df_chats["label_sentiment"].map(SENTIMENT_MAP)
df_chats["label_critical"]  = df_chats["label_critical"].map(CRITICAL_MAP)

print(f"Chats geladen: {len(df_chats)}")
df_chats[["chat_id", "label_task", "label_sentiment", "label_critical"]].head()

Chats geladen: 30


,chat_id,label_task,label_sentiment,label_critical
0,1,Informationssuche und Verständnis,Neutral,Nein
1,2,Technische und analytische Unterstützung,Neutral,Ja
2,3,Technische und analytische Unterstützung,Neutral,Nein
3,4,Technische und analytische Unterstützung,Neutral,Ja
4,5,Informationssuche und Verständnis,Neutral,Nein


## 3. API Key & Labels

## 4. Pydantic Schemas & Labels

In [ ]:
from pydantic import BaseModel, field_validator
from typing import List, Optional

LABELS_TASK = [
    "Informationssuche und Verständnis",
    "Schreiben und Textarbeit",
    "Praktische Unterstützung und Strukturierung",
    "Technische und analytische Unterstützung",
    "Lernen und Prüfungsvorbereitung",
]
LABELS_SENTIMENT = ["Unfreundlich", "Neutral", "Freundlich"]
LABELS_CRITICAL  = ["Ja", "Nein"]

class ChatExample(BaseModel):
    text:             str
    label_task:       Optional[str] = None
    label_sentiment:  Optional[str] = None
    label_critical:   Optional[str] = None

    @field_validator("text")
    @classmethod
    def text_not_empty(cls, v):
        if not v.strip():
            raise ValueError("Text darf nicht leer sein")
        return v.strip()

# Beispiele aus DataFrame bauen
examples: List[ChatExample] = []
errors = []
for _, row in df_chats.iterrows():
    try:
        examples.append(ChatExample(
            text            = str(row["text"]),
            label_task      = str(row["label_task"]),
            label_sentiment = str(row["label_sentiment"]),
            label_critical  = str(row["label_critical"]),
        ))
    except Exception as e:
        errors.append({"row": row.to_dict(), "error": str(e)})

print(f"Gültige Beispiele: {len(examples)} | Übersprungen: {len(errors)}")

Gültige Beispiele: 30 | Übersprungen: 0


## 6. Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Separate Prompts ──────────────────────────────────────────────────────────
prompt_task = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere das folgende Chat-Log und bestimme,
welcher Aufgabentyp vorliegt.

Aufgabentypen:
(1) Informationssuche und Verständnis – z.B. Literaturhinweise, Begriffe erklären, Verständnisfragen
(2) Schreiben und Textarbeit – z.B. Feedback, Gliederungen, Zusammenfassungen, Übersetzungen
(3) Praktische Unterstützung und Strukturierung – z.B. Ideen entwickeln, Projekte strukturieren, Planung
(4) Technische und analytische Unterstützung – z.B. Code, Datenanalyse, Statistik (R/SPSS/Stata)
(5) Lernen und Prüfungsvorbereitung – z.B. Prüfungsvorbereitung, Lernfragen, Karteikarten

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_TASK))

prompt_sentiment = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere den Ton des folgenden Chat-Logs.

Bewerte den Ton als:
- Unfreundlich: abweisend, fordernd, ungeduldig
- Neutral: sachlich, nüchtern, aufgabenorientiert
- Freundlich: höflich, wertschätzend, kooperativ

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_SENTIMENT))

prompt_critical = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem. Analysiere das folgende Chat-Log.

Fragt die Person ChatGPT nach Hinweisen zur Überprüfung der Antwort,
z.B. Quellen, Belege, Gegenargumente oder einer Prüfung auf mögliche Fehler?

Antworte NUR mit EINEM der folgenden Labels (exakt so):
{labels}"""),
    ("human", "Chat-Log:\n{text}\n\nLabel:"),
]).partial(labels=" | ".join(LABELS_CRITICAL))

# ── Kombinierter Prompt ───────────────────────────────────────────────────────
prompt_combined = ChatPromptTemplate.from_messages([
    ("system", """Du bist ein Klassifikationssystem für akademische Chat-Logs.
Analysiere das folgende Chat-Log und klassifiziere es nach drei Kriterien.

KRITERIUM 1 – Aufgabentyp:
(1) Informationssuche und Verständnis – z.B. Literaturhinweise, Begriffe erklären
(2) Schreiben und Textarbeit – z.B. Feedback, Gliederungen, Übersetzungen
(3) Praktische Unterstützung und Strukturierung – z.B. Ideen entwickeln, Planung
(4) Technische und analytische Unterstützung – z.B. Code, Datenanalyse, R/SPSS
(5) Lernen und Prüfungsvorbereitung – z.B. Prüfungsvorbereitung, Karteikarten

KRITERIUM 2 – Ton/Sentiment:
- Unfreundlich: abweisend, fordernd, ungeduldig
- Neutral: sachlich, nüchtern, aufgabenorientiert
- Freundlich: höflich, wertschätzend, kooperativ

KRITERIUM 3 – Kritische Überprüfung:
Fragt die Person nach Quellen, Belegen, Gegenargumenten oder Fehlerprüfung?
- Ja
- Nein

Antworte NUR in diesem exakten Format (drei Zeilen, nichts anderes):
TASK: <Label>
SENTIMENT: <Label>
CRITICAL: <Label>"""),
    ("human", "Chat-Log:\n{text}"),
])

print("Prompts definiert")

Prompts definiert


In [ ]:
!pip install -q langchain langchain-openai langchain-huggingface transformers accelerate

## 7. Modelle laden

In [ ]:
import torch
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def load_local_llm(model_id: str, max_new_tokens: int = 32) -> HuggingFacePipeline:
    print(f"GPU verfügbar: {torch.cuda.is_available()}")  # sollte True sein
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,  # float16 auf GPU
        device_map="auto",          # landet automatisch auf der T4
    )
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    return HuggingFacePipeline(pipeline=pipe)

def make_chain(llm, prompt):
    return prompt | llm | StrOutputParser()

llm_registry = {
    #openAI_api
    #"gpt-4o-mini":   ChatOpenAI(model="gpt-4o-mini",        temperature=0),
    #"gpt-4.1":       ChatOpenAI(model="gpt-4.1",            temperature=0),
    #"gpt-5.4-mini":  ChatOpenAI(model="gpt-5.4-mini",       temperature=0),

    ##local models
    "phi-3-mini":   load_local_llm("microsoft/Phi-3-mini-4k-instruct"),
    "mistral-7b":   load_local_llm("mistralai/Mistral-7B-Instruct-v0.3"),
    "llama-3-8b":   load_local_llm("meta-llama/Meta-Llama-3-8B-Instruct"),
    #"gemma-2-9b":    load_local_llm("google/gemma-2-9b-it"),
}

print(f" Modelle bereit: {list(llm_registry.keys())}")

 Modelle bereit: ['gpt-4o-mini', 'gpt-4.1', 'gpt-5.4-mini']


## 8. Evaluierungs-Funktionen

In [ ]:
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def clean_label(raw: str, valid_labels: List[str]) -> str:
    raw = raw.strip()
    for label in valid_labels:               # exakter Match
        if label.lower() == raw.lower():
            return label
    for label in valid_labels:               # Teilstring-Match
        if label.lower() in raw.lower():
            return label
    return "unknown"

def compute_metrics(preds: List[str], golds: List[str], labels: List[str]) -> dict:
    valid = [(p, g) for p, g in zip(preds, golds) if p != "unknown"]
    if not valid:
        return {"error": "Keine gültigen Predictions"}
    p_clean, g_clean = zip(*valid)
    return {
        "accuracy":        round(accuracy_score(g_clean, p_clean), 4),
        "precision_macro": round(precision_score(g_clean, p_clean, average="macro", labels=labels, zero_division=0), 4),
        "recall_macro":    round(recall_score(g_clean, p_clean, average="macro",    labels=labels, zero_division=0), 4),
        "f1_macro":        round(f1_score(g_clean, p_clean, average="macro",        labels=labels, zero_division=0), 4),
        "unknown_rate":    round(1 - len(valid) / len(preds), 4),
        "n_samples":       len(preds),
        "report":          classification_report(g_clean, p_clean, labels=labels, zero_division=0),
    }

# Separat: eine Aufgabe pro Aufruf
def run_separate(llm, examples: List[ChatExample]) -> dict:

    TASKS = [
        {"name": "task_type", "prompt": prompt_task,      "labels": LABELS_TASK,      "gold_field": "label_task"},
        {"name": "sentiment",  "prompt": prompt_sentiment, "labels": LABELS_SENTIMENT, "gold_field": "label_sentiment"},
        {"name": "critical",   "prompt": prompt_critical,  "labels": LABELS_CRITICAL,  "gold_field": "label_critical"},
    ]
    results = {}
    total_latency = []

    for task in TASKS:
        print(f"    Aufgabe: {task['name']}")
        chain = make_chain(llm, task["prompt"])
        preds, golds, latencies = [], [], []

        for i, ex in enumerate(examples):
            gold = getattr(ex, task["gold_field"])
            if gold is None:
                continue
            start = time.time()
            try:
                raw  = chain.invoke({"text": ex.text})
                pred = clean_label(raw, task["labels"])
            except Exception as e:
                print(f"      Fehler bei Beispiel {i}: {e}")
                pred = "unknown"
            latencies.append(time.time() - start)
            preds.append(pred)
            golds.append(gold)

        metrics = compute_metrics(preds, golds, task["labels"])
        metrics["avg_latency_s"] = round(sum(latencies) / len(latencies), 3) if latencies else 0
        results[task["name"]]    = metrics
        total_latency.extend(latencies)
        print(f"      Accuracy: {metrics.get('accuracy')}  F1: {metrics.get('f1_macro')}")

    results["total_avg_latency_s"] = round(sum(total_latency) / len(total_latency), 3)
    return results

#Kombiniert: alle drei Aufgaben in einem Aufruf
def parse_combined(raw: str) -> dict:
    result = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}
    for line in raw.strip().splitlines():
        line = line.strip()
        if line.startswith("TASK:"):
            result["task"]      = clean_label(line.replace("TASK:", "").strip(), LABELS_TASK)
        elif line.startswith("SENTIMENT:"):
            result["sentiment"] = clean_label(line.replace("SENTIMENT:", "").strip(), LABELS_SENTIMENT)
        elif line.startswith("CRITICAL:"):
            result["critical"]  = clean_label(line.replace("CRITICAL:", "").strip(), LABELS_CRITICAL)
    return result

def run_combined(llm, examples: List[ChatExample]) -> dict:
    """Führt einen kombinierten Prompt für alle drei Aufgaben aus."""
    chain = make_chain(llm, prompt_combined)
    preds_task, preds_sentiment, preds_critical = [], [], []
    golds_task, golds_sentiment, golds_critical = [], [], []
    latencies = []

    for i, ex in enumerate(examples):
        start = time.time()
        try:
            raw    = chain.invoke({"text": ex.text})
            parsed = parse_combined(raw)
        except Exception as e:
            print(f"    Fehler bei Beispiel {i}: {e}")
            parsed = {"task": "unknown", "sentiment": "unknown", "critical": "unknown"}

        latencies.append(time.time() - start)
        preds_task.append(parsed["task"])
        preds_sentiment.append(parsed["sentiment"])
        preds_critical.append(parsed["critical"])
        golds_task.append(ex.label_task)
        golds_sentiment.append(ex.label_sentiment)
        golds_critical.append(ex.label_critical)

        if (i + 1) % 5 == 0:
            print(f"    {i+1}/{len(examples)} verarbeitet ...")

    avg_lat = round(sum(latencies) / len(latencies), 3)
    return {
        "task_type": {**compute_metrics(preds_task,      golds_task,      LABELS_TASK),      "avg_latency_s": avg_lat},
        "sentiment": {**compute_metrics(preds_sentiment, golds_sentiment, LABELS_SENTIMENT), "avg_latency_s": avg_lat},
        "critical":  {**compute_metrics(preds_critical,  golds_critical,  LABELS_CRITICAL),  "avg_latency_s": avg_lat},
        "total_avg_latency_s": avg_lat,
    }



## 9. Evaluierung ausführen

In [ ]:
# Ergebnis-Struktur:
# all_results[model_name]["separate" | "combined"][task_name] = metrics
all_results = {}

for model_name, llm in llm_registry.items():
    print(f"\n{'='*60}")
    print(f"Modell: {model_name}")
    print(f"{'='*60}")
    all_results[model_name] = {}

    print("\n  ── Strategie: SEPARAT ──")
    all_results[model_name]["separate"] = run_separate(llm, examples)

    print("\n  ── Strategie: KOMBINIERT ──")
    all_results[model_name]["combined"] = run_combined(llm, examples)

print("\n Evaluierung abgeschlossen")


Modell: gpt-4o-mini

  ── Strategie: SEPARAT ──
    Aufgabe: task_type
      Accuracy: 0.7667  F1: 0.4397
    Aufgabe: sentiment
      Accuracy: 0.6  F1: 0.3125
    Aufgabe: critical
      Accuracy: 0.7  F1: 0.4118

  ── Strategie: KOMBINIERT ──
    5/30 verarbeitet ...
    10/30 verarbeitet ...
    15/30 verarbeitet ...
    20/30 verarbeitet ...
    25/30 verarbeitet ...
    30/30 verarbeitet ...

Modell: gpt-4.1

  ── Strategie: SEPARAT ──
    Aufgabe: task_type
      Accuracy: 0.7333  F1: 0.4086
    Aufgabe: sentiment
      Fehler bei Beispiel 10: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-ipckaFskEsPZJDSOojjRf4Fp on tokens per min (TPM): Limit 30000, Used 28573, Requested 1611. Please try again in 368ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
      Fehler bei Beispiel 25: Error code: 429 - {'error': {'message': 'Rate limit reached for gp

## 10. Ergebnisse & Vergleich

In [ ]:
import pandas as pd

TASK_NAMES = ["task_type", "sentiment", "critical"]
rows = []

for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            rows.append({
                "Modell":       model_name,
                "Strategie":    strategy,
                "Aufgabe":      task_name,
                "Accuracy":     m.get("accuracy"),
                "Precision":    m.get("precision_macro"),
                "Recall":       m.get("recall_macro"),
                "F1 (macro)":   m.get("f1_macro"),
                "Unknown-Rate": m.get("unknown_rate"),
                "Latenz (s)":   m.get("avg_latency_s"),
            })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["Modell", "Aufgabe", "Strategie"])
summary_df

,Modell,Strategie,Aufgabe,Accuracy,Precision,Recall,F1 (macro),Unknown-Rate,Latenz (s)
11,gpt-4.1,combined,critical,0.7600,0.5536,0.5536,0.5536,0.1667,1.572
8,gpt-4.1,separate,critical,0.7857,0.4231,0.4583,0.4400,0.0667,1.259
10,gpt-4.1,combined,sentiment,0.8400,0.6232,0.4841,0.5253,0.1667,1.572
7,gpt-4.1,separate,sentiment,0.7143,0.3750,0.4545,0.4087,0.0667,0.785
9,gpt-4.1,combined,task_type,0.7200,0.3000,0.3346,0.3155,0.1667,1.572
6,gpt-4.1,separate,task_type,0.7333,0.4070,0.4200,0.4086,0.0000,0.452
5,gpt-4o-mini,combined,critical,0.6429,0.4762,0.4696,0.4697,0.0667,0.566
2,gpt-4o-mini,separate,critical,0.7000,0.3889,0.4375,0.4118,0.0000,0.524
4,gpt-4o-mini,combined,sentiment,0.8333,0.7933,0.6389,0.6882,0.0000,0.566
1,gpt-4o-mini,separate,sentiment,0.6000,0.3069,0.4028,0.3125,0.0000,0.470


In [ ]:
#F1-Differenz: Separat vs. Kombiniert
print(f"{'Modell':<20} {'Aufgabe':<12} {'F1 Separat':>11} {'F1 Kombiniert':>14} {'Δ F1':>8}")
print("-" * 70)

for model_name, strategies in all_results.items():
    for task_name in TASK_NAMES:
        f1_sep = strategies["separate"].get(task_name, {}).get("f1_macro", None)
        f1_com = strategies["combined"].get(task_name, {}).get("f1_macro", None)
        if f1_sep is not None and f1_com is not None:
            delta = f1_com - f1_sep
            marker = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
            print(f"{model_name:<20} {task_name:<12} {f1_sep:>11.4f} {f1_com:>14.4f} {marker}{abs(delta):>7.4f}")

Modell               Aufgabe       F1 Separat  F1 Kombiniert     Δ F1
----------------------------------------------------------------------
gpt-4o-mini          task_type         0.4397         0.4025 ▼ 0.0372
gpt-4o-mini          sentiment         0.3125         0.6882 ▲ 0.3757
gpt-4o-mini          critical          0.4118         0.4697 ▲ 0.0579
gpt-4.1              task_type         0.4086         0.3155 ▼ 0.0931
gpt-4.1              sentiment         0.4087         0.5253 ▲ 0.1166
gpt-4.1              critical          0.4400         0.5536 ▲ 0.1136
gpt-5.4-mini         task_type         0.4567         0.5557 ▲ 0.0990
gpt-5.4-mini         sentiment         0.5302         0.4339 ▼ 0.0963
gpt-5.4-mini         critical          0.5200         0.3617 ▼ 0.1583


In [ ]:
#Classification Reports
for model_name, strategies in all_results.items():
    for strategy, tasks in strategies.items():
        for task_name in TASK_NAMES:
            m = tasks.get(task_name, {})
            if "report" in m:
                print(f"\n{'='*60}")
                print(f"{model_name}  |  {strategy}  |  {task_name}")
                print(f"{'='*60}")
                print(m["report"])


gpt-4o-mini  |  separate  |  task_type
                                             precision    recall  f1-score   support

          Informationssuche und Verständnis       0.88      0.78      0.82         9
                   Schreiben und Textarbeit       0.50      0.50      0.50         4
Praktische Unterstützung und Strukturierung       0.00      0.00      0.00         2
   Technische und analytische Unterstützung       0.82      0.93      0.88        15
            Lernen und Prüfungsvorbereitung       0.00      0.00      0.00         0

                                   accuracy                           0.77        30
                                  macro avg       0.44      0.44      0.44        30
                               weighted avg       0.74      0.77      0.75        30


gpt-4o-mini  |  separate  |  sentiment
              precision    recall  f1-score   support

Unfreundlich       0.11      0.50      0.18         2
     Neutral       0.81      0.71      0.76

In [ ]:
import json

summary_df.toDu siehst die Dateien links in der Seitenleiste unter dem Ordner-Symbol 📁._csv("eval_results_v3.csv", index=False)
print("eval_results_v3.csv gespeichert")

export = {
    model: {
        strategy: {
            task: {k: v for k, v in metrics.items() if k != "report"}
            for task, metrics in tasks.items()
            if isinstance(metrics, dict)
        }
        for strategy, tasks in strategies.items()
    }
    for model, strategies in all_results.items()
}
with open("eval_results_v3.json", "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print("eval_results_v3.json gespeichert")

eval_results_v3.csv gespeichert
eval_results_v3.json gespeichert
